# Mini-Lakehouse demo

Query the Iceberg tables with **DuckDB** through the REST catalog. Run `make up` first so the stack and demo producer are live.

This notebook shows three things from the design doc: a live query, the compaction effect, and time travel to an earlier snapshot.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL iceberg; LOAD iceberg;")
# Point DuckDB at the local Iceberg REST catalog backed by MinIO.
con.sql("""
ATTACH '' AS lakehouse (
  TYPE iceberg,
  ENDPOINT 'http://localhost:8181'
);
""")
con.sql("SHOW ALL TABLES").show()

## Daily active players (headline analytics query)

In [ ]:
con.sql("""
SELECT date_trunc('day', event_time) AS day,
       count(DISTINCT player_id)      AS daily_active_players,
       count(*)                       AS events
FROM lakehouse.bronze.events
GROUP BY 1
ORDER BY 1
""").show()

## Time travel

Every ingestor commit is a snapshot carrying its Kafka offsets. Read an earlier snapshot to see the table as it was.

In [ ]:
snaps = con.sql("SELECT * FROM lakehouse.bronze.events.snapshots ORDER BY committed_at").fetchall()
print(f"{len(snaps)} snapshots")
# Example: count rows as of the first snapshot id.
# con.sql("SELECT count(*) FROM lakehouse.bronze.events AT (SNAPSHOT => <id>)").show()